## Example 1: Basic pattern matching

In [ ]:
import pathspec

# Example 1: Basic pathspec pattern matching
print("Example 1: Basic pathspec pattern matching")

try:
    # Create a pathspec from gitignore-style patterns
    patterns = [
        '*.py',        # Include Python files
        '!test_*.py',  # Exclude test files
        'build/',      # Include build directory
        '*.log',       # Include log files
        '!debug.log'   # Exclude debug.log specifically
    ]
    
    spec = pathspec.PathSpec.from_lines('gitwildmatch', patterns)
    print(f"PathSpec created with {len(patterns)} patterns")
    
    # Test various file paths
    test_files = [
        'main.py',
        'test_main.py', 
        'utils.py',
        'test_utils.py',
        'config.json',
        'app.log',
        'debug.log',
        'build/output.txt',
        'docs/readme.md'
    ]
    
    print("\nFile matching results:")
    print("=" * 40)
    for file_path in test_files:
        match = spec.match_file(file_path)
        status = "✓ MATCH" if match else "✗ NO MATCH"
        print(f"  {file_path:20} -> {status}")
    
    print("pathspec example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pathspec example 1 failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Advanced pattern matching and file filtering

In [ ]:
import pathspec
import os

# Example 2: Advanced pattern matching and file filtering
print("Example 2: Advanced pattern matching and file filtering")

try:
    # Create complex patterns like those in .gitignore files
    gitignore_patterns = [
        '__pycache__/',
        '*.pyc',
        '*.pyo', 
        '*.pyd',
        '.venv/',
        'venv/',
        '.env',
        'node_modules/',
        '*.log',
        'dist/',
        'build/',
        '!dist/README.txt'  # Exception: keep this specific file
    ]
    
    spec = pathspec.PathSpec.from_lines('gitwildmatch', gitignore_patterns)
    
    # Test with a realistic file structure
    test_structure = [
        'src/main.py',
        'src/__pycache__/main.cpython-39.pyc',
        'tests/test_main.py',
        'venv/lib/python3.9/site-packages/',
        'node_modules/package/index.js',
        '.env',
        'config.json',
        'app.log',
        'debug.log',
        'dist/app.js',
        'dist/README.txt',  # Should NOT be ignored due to !dist/README.txt
        'build/output/final.exe'
    ]
    
    print("Gitignore-style filtering:")
    print("=" * 45)
    for path in test_structure:
        ignored = spec.match_file(path)
        status = "IGNORED" if ignored else "TRACKED"
        print(f"  {path:35} -> {status}")
    
    # Show filtered results (files that would be tracked)
    tracked_files = [path for path in test_structure if not spec.match_file(path)]
    print(f"\nTracked files ({len(tracked_files)}):")
    for file in tracked_files:
        print(f"  ✓ {file}")
    
    print("pathspec example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pathspec example 2 failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: matching files in a directory tree

In [ ]:
import os
import tempfile
import pathspec

# Example 3: walk a real directory tree and list the files a .gitignore would ignore
print("Example 3: match_tree_files over a temporary directory")

spec = pathspec.GitIgnoreSpec.from_lines([
    "__pycache__/",
    "*.log",
    "!keep.log",   # re-include one log file
    "build/",
])
layout = [
    "README.md",
    "src/app.py",
    "src/__pycache__/app.cpython-312.pyc",
    "logs/run.log",
    "logs/keep.log",
    "build/out.bin",
]
with tempfile.TemporaryDirectory() as root:
    for relative in layout:
        path = os.path.join(root, *relative.split("/"))
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w") as handle:
            handle.write("x")
    # match_tree_files yields the matching files as paths relative to the root
    ignored = sorted(path.replace(os.sep, "/") for path in spec.match_tree_files(root))

expected = ["build/out.bin", "logs/run.log", "src/__pycache__/app.cpython-312.pyc"]
assert ignored == expected, ignored
for relative in layout:
    print(f"  {relative:40} -> {'IGNORED' if relative in ignored else 'TRACKED'}")
print("Example 3 finished")

## Example 4: which pattern decided a path

In [ ]:
import pathspec

# Example 4: check_file reports which pattern decided a path, not just whether it matched
print("Example 4: check_file and the deciding pattern")

patterns = ["*.log", "!important.log", "tmp/"]
spec = pathspec.GitIgnoreSpec.from_lines(patterns)

# include is True (ignored), False (re-included by a "!" pattern) or None (no pattern matched);
# index is the position of the last pattern that matched.
for path, include, index in (
    ("debug.log", True, 0),
    ("important.log", False, 1),
    ("tmp/cache.bin", True, 2),
    ("notes.txt", None, None),
):
    result = spec.check_file(path)
    assert (result.file, result.include, result.index) == (path, include, index), result
    deciding = patterns[result.index] if result.index is not None else "no pattern"
    print(f"  {path:15} include={result.include!s:5} decided by {deciding!r}")
print("Example 4 finished")

## Example 5: combining specs and filtering a path list

In [ ]:
import pathspec

# Example 5: combine two specs with + and filter a list of paths with match_files
print("Example 5: combining specs and filtering paths")

python_junk = pathspec.GitIgnoreSpec.from_lines(["__pycache__/", "*.pyc"])
editor_junk = pathspec.GitIgnoreSpec.from_lines([".vscode/", "*.swp"])
combined = python_junk + editor_junk
assert len(combined) == len(python_junk) + len(editor_junk) == 4, len(combined)

paths = [
    "pkg/__init__.py",
    "pkg/__pycache__/__init__.cpython-312.pyc",
    "pkg/module.pyc",
    ".vscode/settings.json",
    "pkg/.module.py.swp",
    "setup.cfg",
]
assert sorted(python_junk.match_files(paths)) == ["pkg/__pycache__/__init__.cpython-312.pyc", "pkg/module.pyc"]
ignored = sorted(combined.match_files(paths))
assert ignored == [".vscode/settings.json", "pkg/.module.py.swp",
                   "pkg/__pycache__/__init__.cpython-312.pyc", "pkg/module.pyc"], ignored
kept = [path for path in paths if path not in ignored]
assert kept == ["pkg/__init__.py", "setup.cfg"], kept
print("ignored:", ignored)
print("kept:", kept)
print("Example 5 finished")